In [ ]:
# ArcGIS Pro (ArcPy) full script
# Load MK results + Thiessen location CSV, rename ID_1 -> ID, join, filter PRCP where trend != "no trend",
# create point layer, add to Map2026_march, and label those points with "*" in black, 8 pt, halo (outline) 2 pt.
#
# IMPORTANT: edit aprx_path and out_gdb paths before running.

import os
import arcpy

# -----------------------------
# USER INPUTS (EDIT THESE)
# -----------------------------
mk_csv_updated = r"./data_intermediate\mk_two_PRCP_TMAX_TMIN_AI.csv"
loc_csv_updated = r"./data_raw\Thiessen_Polygons_100yrs.csv"
out_gdb_updated = r"./arcgis_work\mk_map.gdb"
mk_csv_spei_updated = r"./data_intermediate_spei_monthly\mk_two_spei_monthly.csv"
df_stats_url = r"./data_intermediate\ai_station_stats.csv"

df_trend = pd.read_csv(mk_csv_updated)
df_loc = pd.read_csv(loc_csv_updated)

df_loc_needed = df_loc[["ID_1","Lat","Lon"]]
df_loc_needed.rename(columns={"ID_1":"ID"}, inplace = True)

df_trend_updated = df_trend.merge(df_loc_needed, on="ID", how="left")

In [ ]:
import os
import arcpy
import pandas as pd

arcpy.env.overwriteOutput = True


def _ensure_gdb(out_gdb):
    out_folder = os.path.dirname(out_gdb)
    if not os.path.exists(out_folder):
        os.makedirs(out_folder)
    if not arcpy.Exists(out_gdb):
        arcpy.management.CreateFileGDB(out_folder, os.path.basename(out_gdb))


def _safe_delete(path):
    if arcpy.Exists(path):
        arcpy.management.Delete(path)


def _get_map(aprx, map_name):
    maps = aprx.listMaps(map_name)
    if not maps:
        raise RuntimeError("Map named '{}' not found.".format(map_name))
    return maps[0]


def _remove_layer_by_name(map_obj, layer_name):
    for lyr in list(map_obj.listLayers()):
        try:
            if lyr.name == layer_name:
                map_obj.removeLayer(lyr)
        except Exception:
            pass


def _remove_layers_ending_with(map_obj, suffix):
    suffix = suffix.lower()
    for lyr in list(map_obj.listLayers()):
        try:
            if lyr.name.lower().endswith(suffix):
                map_obj.removeLayer(lyr)
        except Exception:
            pass


def _remove_layers_containing(map_obj, tokens):
    tokens = [t.lower() for t in tokens]
    for lyr in list(map_obj.listLayers()):
        try:
            nm = lyr.name.lower()
            if any(tok in nm for tok in tokens):
                map_obj.removeLayer(lyr)
        except Exception:
            pass


def _cleanup_tmp_fc_and_layers(map_obj, out_gdb, tmp_fc_name):
    _remove_layer_by_name(map_obj, tmp_fc_name)
    _remove_layers_ending_with(map_obj, "_tmp")
    _safe_delete(os.path.join(out_gdb, tmp_fc_name))


def _build_points_for_window(
    df_trend_updated,
    out_gdb,
    out_folder,
    variable,
    window_type,
    nice_layer_name,
    raw_points_fc_name,
    table_name,
    map_obj,
    xy_epsg=4326
):
    # Filter
    df_plot = df_trend_updated[
        (df_trend_updated["variable"] == variable) &
        (df_trend_updated["window_type"] == window_type) &
        (df_trend_updated["trend"] != "no trend")
    ].copy()

    df_plot = df_plot.dropna(subset=["Lat", "Lon"]).copy()

    # Export filtered attribute table
    temp_csv = os.path.join(out_folder, "__temp_{}.csv".format(table_name))
    df_plot.to_csv(temp_csv, index=False)

    table_path = os.path.join(out_gdb, table_name)
    _safe_delete(table_path)
    arcpy.conversion.TableToTable(temp_csv, out_gdb, table_name)

    # Build temp points
    sr = arcpy.SpatialReference(xy_epsg)

    raw_points_fc = os.path.join(out_gdb, raw_points_fc_name)
    tmp_fc_name = raw_points_fc_name + "_tmp"
    tmp_fc = os.path.join(out_gdb, tmp_fc_name)

    _safe_delete(tmp_fc)

    arcpy.management.XYTableToPoint(
        in_table=table_path,
        out_feature_class=tmp_fc,
        x_field="Lon",
        y_field="Lat",
        coordinate_system=sr
    )

    # Update stable FC
    if arcpy.Exists(raw_points_fc):
        arcpy.management.DeleteRows(raw_points_fc)
        arcpy.management.Append(tmp_fc, raw_points_fc, "NO_TEST")
    else:
        arcpy.management.CopyFeatures(tmp_fc, raw_points_fc)

    # Add to map as a nice named layer (only)
    _remove_layer_by_name(map_obj, nice_layer_name)
    lyr = map_obj.addDataFromPath(raw_points_fc)
    try:
        lyr.name = nice_layer_name
    except Exception:
        pass

    # Cleanup temp FC and temp CSV
    _cleanup_tmp_fc_and_layers(map_obj, out_gdb, tmp_fc_name)

    try:
        os.remove(temp_csv)
    except Exception:
        pass

    return {
        "layer_name": nice_layer_name,
        "raw_fc_name": raw_points_fc_name,
        "points_fc": raw_points_fc,
        "table_path": table_path,
        "n_rows": int(df_plot.shape[0])
    }


def main():
    mk_csv = mk_csv_updated
    loc_csv = loc_csv_updated
    out_gdb = out_gdb_updated
    map_name = "Map2026_march"
    xy_epsg = 4326

    # Variables to generate layers for 
    variables = ["PRCP", "TMAX", "TMIN", "CSDI", "WSDI", "PCI", "SDII","AI"]
    #variables = ["SDII","AI"]

    _ensure_gdb(out_gdb)
    out_folder = os.path.dirname(out_gdb)

    aprx = arcpy.mp.ArcGISProject("CURRENT")
    m = _get_map(aprx, map_name)

    # Clean up any leftover temp layers from older runs
    _remove_layers_ending_with(m, "_tmp")

    # Build merged dataframe once
    df_trend = pd.read_csv(mk_csv)
    df_loc = pd.read_csv(loc_csv)

    df_loc_needed = df_loc[["ID_1", "Lat", "Lon"]].copy()
    df_loc_needed.rename(columns={"ID_1": "ID"}, inplace=True)

    df_trend_updated = df_trend.merge(df_loc_needed, on="ID", how="left")

    # Build layers for each variable
    for variable in variables:
        var_tag = variable.lower()

        raw_fc_100 = "{}_stations_100yrs_pts".format(var_tag)
        raw_fc_30 = "{}_stations_30yrs_pts".format(var_tag)

        nice_100 = "{} stations (100 yrs)".format(variable)
        nice_30 = "{} stations (30 yrs)".format(variable)

        tbl_100 = "{}_stations_100yrs_tbl".format(var_tag)
        tbl_30 = "{}_stations_30yrs_tbl".format(var_tag)

        res_100 = _build_points_for_window(
            df_trend_updated, out_gdb, out_folder,
            variable, "full_100yr",
            nice_100, raw_fc_100, tbl_100, m, xy_epsg
        )

        res_30 = _build_points_for_window(
            df_trend_updated, out_gdb, out_folder,
            variable, "recent_30yr",
            nice_30, raw_fc_30, tbl_30, m, xy_epsg
        )

        # Remove the raw FC layers if ArcGIS added them as separate layers
        _remove_layers_containing(m, [raw_fc_100, raw_fc_30])

        # Ensure only the nice layers exist (re-add from the FC)
        _remove_layer_by_name(m, nice_100)
        lyr100 = m.addDataFromPath(os.path.join(out_gdb, raw_fc_100))
        try:
            lyr100.name = nice_100
        except Exception:
            pass

        _remove_layer_by_name(m, nice_30)
        lyr30 = m.addDataFromPath(os.path.join(out_gdb, raw_fc_30))
        try:
            lyr30.name = nice_30
        except Exception:
            pass

        print("DONE:", variable)
        print(res_100)
        print(res_30)
        print("Layers expected:", nice_100, "and", nice_30)

    print("ALL VARIABLES DONE")
    print("Ctrl+S in ArcGIS Pro if you want to keep the layers.")


if __name__ == "__main__":
    main()


In [ ]:


# SPEI

def _ensure_gdb(out_gdb):
    out_folder = os.path.dirname(out_gdb)
    if not os.path.exists(out_folder):
        os.makedirs(out_folder)
    if not arcpy.Exists(out_gdb):
        arcpy.management.CreateFileGDB(out_folder, os.path.basename(out_gdb))


def _safe_delete(path):
    try:
        if arcpy.Exists(path):
            arcpy.management.Delete(path)
    except Exception:
        pass


def _get_map(aprx, map_name):
    maps = aprx.listMaps(map_name)
    if not maps:
        raise RuntimeError("Map named '{}' not found.".format(map_name))
    return maps[0]


def _remove_layer_by_name(map_obj, layer_name):
    for lyr in list(map_obj.listLayers()):
        try:
            if lyr.name == layer_name:
                map_obj.removeLayer(lyr)
        except Exception:
            pass


def _remove_layers_ending_with(map_obj, suffix):
    suffix = suffix.lower()
    for lyr in list(map_obj.listLayers()):
        try:
            if lyr.name.lower().endswith(suffix):
                map_obj.removeLayer(lyr)
        except Exception:
            pass


def _remove_layers_containing(map_obj, tokens):
    tokens = [t.lower() for t in tokens]
    for lyr in list(map_obj.listLayers()):
        try:
            nm = lyr.name.lower()
            if any(tok in nm for tok in tokens):
                map_obj.removeLayer(lyr)
        except Exception:
            pass


def _cleanup_tmp_fc_and_layers(map_obj, out_gdb, tmp_fc_name):
    _remove_layer_by_name(map_obj, tmp_fc_name)
    _remove_layers_ending_with(map_obj, "_tmp")
    _safe_delete(os.path.join(out_gdb, tmp_fc_name))


def _build_points_for_window(
    df_trend_updated,
    out_gdb,
    out_folder,
    variable,
    window_type,
    nice_layer_name,
    raw_points_fc_name,
    table_name,
    map_obj,
    xy_epsg=4326,
    variable_col="variable",
    window_col="window_type",
    trend_col="trend",
    trend_keep_value="no trend",
):
    # Filter (keeps all rows except trend == "no trend")
    df_plot = df_trend_updated[
        (df_trend_updated[variable_col] == variable) &
        (df_trend_updated[window_col] == window_type) &
        (df_trend_updated[trend_col] != trend_keep_value)
    ].copy()

    df_plot = df_plot.dropna(subset=["Lat", "Lon"]).copy()

    # Export filtered attribute table
    temp_csv = os.path.join(out_folder, "__temp_{}.csv".format(table_name))
    df_plot.to_csv(temp_csv, index=False)

    table_path = os.path.join(out_gdb, table_name)
    _safe_delete(table_path)
    arcpy.conversion.TableToTable(temp_csv, out_gdb, table_name)

    # Build temp points
    sr = arcpy.SpatialReference(xy_epsg)

    raw_points_fc = os.path.join(out_gdb, raw_points_fc_name)
    tmp_fc_name = raw_points_fc_name + "_tmp"
    tmp_fc = os.path.join(out_gdb, tmp_fc_name)

    _safe_delete(tmp_fc)

    arcpy.management.XYTableToPoint(
        in_table=table_path,
        out_feature_class=tmp_fc,
        x_field="Lon",
        y_field="Lat",
        coordinate_system=sr
    )

    # Update stable FC
    if arcpy.Exists(raw_points_fc):
        arcpy.management.DeleteRows(raw_points_fc)
        arcpy.management.Append(tmp_fc, raw_points_fc, "NO_TEST")
    else:
        arcpy.management.CopyFeatures(tmp_fc, raw_points_fc)

    # Add to map as a nice named layer (only)
    _remove_layer_by_name(map_obj, nice_layer_name)
    lyr = map_obj.addDataFromPath(raw_points_fc)
    try:
        lyr.name = nice_layer_name
    except Exception:
        pass

    # Cleanup temp FC and temp CSV
    _cleanup_tmp_fc_and_layers(map_obj, out_gdb, tmp_fc_name)

    try:
        os.remove(temp_csv)
    except Exception:
        pass

    return {
        "layer_name": nice_layer_name,
        "raw_fc_name": raw_points_fc_name,
        "points_fc": raw_points_fc,
        "table_path": table_path,
        "n_rows": int(df_plot.shape[0])
    }


def main():
    # ---- SPEI inputs ----
    mk_csv = mk_csv_spei_updated
    loc_csv = loc_csv_updated
    out_gdb = out_gdb_updated
    map_name = "Map2026_march"
    xy_epsg = 4326

    # SPEI variables in your file
    variables = ["spei1", "spei3", "spei6", "spei12"]

    _ensure_gdb(out_gdb)
    out_folder = os.path.dirname(out_gdb)

    aprx = arcpy.mp.ArcGISProject("CURRENT")
    m = _get_map(aprx, map_name)

    # Clean up any leftover temp layers from older runs
    _remove_layers_ending_with(m, "_tmp")

    # Build merged dataframe once
    df_trend = pd.read_csv(mk_csv)
    df_loc = pd.read_csv(loc_csv)

    df_loc_needed = df_loc[["ID_1", "Lat", "Lon"]].copy()
    df_loc_needed.rename(columns={"ID_1": "ID"}, inplace=True)

    df_trend_updated = df_trend.merge(df_loc_needed, on="ID", how="left")

    # Build layers for each variable
    for variable in variables:
        var_tag = variable.lower()

        # keep your same naming pattern used earlier
        raw_fc_100 = "{}_stations_100yrs_pts".format(var_tag)
        raw_fc_30 = "{}_stations_30yrs_pts".format(var_tag)

        # nice layer names to match your other layers style
        # (spei1 -> SPEI1 stations (30 yrs))
        nice_var = var_tag.upper()  # SPEI1, SPEI3, ...
        nice_100 = "{} stations (100 yrs)".format(nice_var)
        nice_30 = "{} stations (30 yrs)".format(nice_var)

        tbl_100 = "{}_stations_100yrs_tbl".format(var_tag)
        tbl_30 = "{}_stations_30yrs_tbl".format(var_tag)

        res_100 = _build_points_for_window(
            df_trend_updated, out_gdb, out_folder,
            variable, "full_100yr",
            nice_100, raw_fc_100, tbl_100, m, xy_epsg
        )

        res_30 = _build_points_for_window(
            df_trend_updated, out_gdb, out_folder,
            variable, "recent_30yr",
            nice_30, raw_fc_30, tbl_30, m, xy_epsg
        )

        # Remove any raw FC layers if ArcGIS added them as separate layers
        _remove_layers_containing(m, [raw_fc_100, raw_fc_30])

        # Ensure only the nice layers exist (re-add from the FC)
        _remove_layer_by_name(m, nice_100)
        lyr100 = m.addDataFromPath(os.path.join(out_gdb, raw_fc_100))
        try:
            lyr100.name = nice_100
        except Exception:
            pass

        _remove_layer_by_name(m, nice_30)
        lyr30 = m.addDataFromPath(os.path.join(out_gdb, raw_fc_30))
        try:
            lyr30.name = nice_30
        except Exception:
            pass

        print("DONE:", variable)
        print(res_100)
        print(res_30)
        print("Layers expected:", nice_100, "and", nice_30)

    print("ALL SPEI VARIABLES DONE")
    print("Ctrl+S in ArcGIS Pro if you want to keep the layers.")


if __name__ == "__main__":
    main()


In [ ]:
# FULL FLOW FOR AVERAGE 30 YR, 100 YR AND DIFFERENCES

import os
import arcpy
import pandas as pd

arcpy.env.overwriteOutput = True


# ============================================================
# EXISTING HELPERS
# Keep these exactly as you already use them
# ============================================================
def ensure_gdb(out_gdb):
    folder = os.path.dirname(out_gdb)
    if not os.path.exists(folder):
        os.makedirs(folder)
    if not arcpy.Exists(out_gdb):
        arcpy.management.CreateFileGDB(folder, os.path.basename(out_gdb))


def safe_delete(path):
    try:
        if arcpy.Exists(path):
            arcpy.management.Delete(path)
    except:
        pass


def get_map(map_name="Map2026_march", aprx_path=None):
    aprx = arcpy.mp.ArcGISProject("CURRENT" if aprx_path is None else aprx_path)
    maps = aprx.listMaps(map_name)
    if not maps:
        raise RuntimeError(f"Map named '{map_name}' not found.")
    return aprx, maps[0]


def find_layer_by_name(map_obj, layer_name):
    for lyr in map_obj.listLayers():
        if lyr.name == layer_name:
            return lyr
    return None


def remove_layer_if_exists(map_obj, layer_name):
    lyr = find_layer_by_name(map_obj, layer_name)
    if lyr:
        map_obj.removeLayer(lyr)


def make_stats_df(df_stats_url):
    df_stats = pd.read_csv(df_stats_url)
    drop_cols = [c for c in ["domainName", "Area"] if c in df_stats.columns]
    return df_stats.drop(columns=drop_cols, errors="ignore")


def merge_df_with_locations(df_in, loc_csv, id_col="ID", loc_id_col="ID_1", lat_col="Lat", lon_col="Lon"):
    df_loc = pd.read_csv(loc_csv)
    df_loc_needed = df_loc[[loc_id_col, lat_col, lon_col]].copy()
    df_loc_needed.rename(columns={loc_id_col: id_col}, inplace=True)
    return df_in.merge(df_loc_needed, on=id_col, how="left")


def df_to_gdb_table(df, out_gdb, table_name, temp_csv_dir=None):
    ensure_gdb(out_gdb)
    temp_csv_dir = temp_csv_dir or os.path.dirname(out_gdb)
    temp_csv = os.path.join(temp_csv_dir, f"__temp_{table_name}.csv")

    df.to_csv(temp_csv, index=False)

    table_path = os.path.join(out_gdb, table_name)
    safe_delete(table_path)
    arcpy.conversion.TableToTable(temp_csv, out_gdb, table_name)

    try:
        os.remove(temp_csv)
    except:
        pass

    return table_path


def table_to_points_xy(table_path, out_gdb, fc_name, x_field="Lon", y_field="Lat", xy_epsg=4326):
    ensure_gdb(out_gdb)
    out_fc = os.path.join(out_gdb, fc_name)
    safe_delete(out_fc)

    sr = arcpy.SpatialReference(xy_epsg)
    arcpy.management.XYTableToPoint(
        in_table=table_path,
        out_feature_class=out_fc,
        x_field=x_field,
        y_field=y_field,
        coordinate_system=sr
    )
    return out_fc


def run_idw(points_fc, value_field, out_raster, cell_size=None, power=2, search_radius=None):
    arcpy.CheckOutExtension("Spatial")
    from arcpy.sa import Idw

    kwargs = {}
    if cell_size is not None:
        kwargs["cell_size"] = cell_size
    if power is not None:
        kwargs["power"] = power
    if search_radius is not None:
        kwargs["search_radius"] = search_radius

    ras = Idw(in_point_features=points_fc, z_field=value_field, **kwargs)
    ras.save(out_raster)
    return out_raster


def mask_raster_cleanly(in_raster, mask_fc, out_raster):
    arcpy.CheckOutExtension("Spatial")
    from arcpy.sa import ExtractByMask

    masked = ExtractByMask(in_raster, mask_fc)
    masked.save(out_raster)
    return out_raster


def add_to_map(map_obj, path, layer_name):
    remove_layer_if_exists(map_obj, layer_name)
    lyr = map_obj.addDataFromPath(path)
    try:
        lyr.name = layer_name
    except:
        pass
    return lyr


# ============================================================
# NEW GENERIC SINGLE-RUN BUILDER
# ============================================================
def build_idw_from_field(
    var_name,
    df_stats_url,
    loc_csv,
    out_gdb,
    value_field,
    final_raster_dataset_name,
    final_map_layer_name,
    map_name="Map2026_march",
    clip_layer_name="Eco region",
    xy_epsg=4326,
    cell_size=None,
    power=2,
    clip_output=True
):
    ensure_gdb(out_gdb)
    aprx, m = get_map(map_name=map_name)

    df_stats_needed = make_stats_df(df_stats_url)
    df_merged = merge_df_with_locations(df_stats_needed, loc_csv)

    if value_field not in df_merged.columns:
        raise ValueError(f"Field '{value_field}' not found in {df_stats_url}")

    df_merged = df_merged.dropna(subset=["Lat", "Lon", value_field]).copy()

    run_tag = final_raster_dataset_name.lower()

    tbl_name = f"__{run_tag}_tbl"
    pts_name = f"__{run_tag}_pts"
    raw_idw_name = f"__{run_tag}_raw"

    tbl_path = df_to_gdb_table(df_merged, out_gdb, tbl_name)
    pts_fc = table_to_points_xy(tbl_path, out_gdb, pts_name, x_field="Lon", y_field="Lat", xy_epsg=xy_epsg)

    raw_idw_raster = os.path.join(out_gdb, raw_idw_name)
    final_raster = os.path.join(out_gdb, final_raster_dataset_name)

    safe_delete(raw_idw_raster)
    safe_delete(final_raster)

    clip_fc_path = None
    if clip_output:
        clip_lyr = find_layer_by_name(m, clip_layer_name)
        if clip_lyr is None:
            raise RuntimeError(f"Could not find clip layer '{clip_layer_name}' in the map.")

        clip_fc_path = clip_lyr.dataSource
        desc = arcpy.Describe(clip_fc_path)
        arcpy.env.extent = desc.extent
        arcpy.env.mask = clip_fc_path

    run_idw(pts_fc, value_field, raw_idw_raster, cell_size=cell_size, power=power)

    if clip_output:
        mask_raster_cleanly(raw_idw_raster, clip_fc_path, final_raster)
    else:
        arcpy.management.CopyRaster(raw_idw_raster, final_raster)

    add_to_map(m, final_raster, final_map_layer_name)

    for nm in [tbl_name, pts_name]:
        remove_layer_if_exists(m, nm)

    safe_delete(os.path.join(out_gdb, tbl_name))
    safe_delete(os.path.join(out_gdb, pts_name))
    safe_delete(raw_idw_raster)

    arcpy.env.extent = None
    arcpy.env.mask = None

    return {
        "variable": var_name,
        "value_field": value_field,
        "final_raster": final_raster,
        "final_layer_name": final_map_layer_name
    }


# ============================================================
# FIELD / NAME SPECS
# This gives the four figures per variable
# ============================================================
def make_four_output_specs(var_name):
    return [
        {
            "value_field": f"{var_name}_recent_mean",
            "raster_name": f"{var_name}_30yrs",
            "layer_name": f"{var_name} (30 yrs)"
        },
        {
            "value_field": f"{var_name}_full_mean",
            "raster_name": f"{var_name}_100yrs",
            "layer_name": f"{var_name} (100 yrs)"
        },
        {
            "value_field": f"{var_name}_recent_minus_penult",
            "raster_name": f"{var_name}_rmp",
            "layer_name": f"{var_name} (rmp)"
        },
        {
            "value_field": f"{var_name}_recent_minus_full",
            "raster_name": f"{var_name}_rmf",
            "layer_name": f"{var_name} (rmf)"
        }
    ]


# ============================================================
# RUN ALL FOUR MAPS FOR ONE VARIABLE
# ============================================================
def build_all_four_for_variable(
    var_name,
    df_stats_url,
    loc_csv,
    out_gdb,
    map_name="Map2026_march",
    clip_layer_name="Eco region",
    xy_epsg=4326,
    cell_size=None,
    power=2,
    clip_output=True
):
    specs = make_four_output_specs(var_name)
    var_results = {}

    for spec in specs:
        value_field = spec["value_field"]
        raster_name = spec["raster_name"]
        layer_name = spec["layer_name"]

        print(f"    Creating {layer_name} from field '{value_field}'")

        res = build_idw_from_field(
            var_name=var_name,
            df_stats_url=df_stats_url,
            loc_csv=loc_csv,
            out_gdb=out_gdb,
            value_field=value_field,
            final_raster_dataset_name=raster_name,
            final_map_layer_name=layer_name,
            map_name=map_name,
            clip_layer_name=clip_layer_name,
            xy_epsg=xy_epsg,
            cell_size=cell_size,
            power=power,
            clip_output=clip_output
        )

        var_results[value_field] = res

    return var_results


# ============================================================
# RUN ALL VARIABLES
# ============================================================
def build_all_variables_all_four(
    variable_csv_dict,
    loc_csv,
    out_gdb,
    map_name="Map2026_march",
    clip_layer_name="Eco region",
    xy_epsg=4326,
    cell_size=None,
    power=2,
    clip_output=True
):
    all_results = {}
    failed_runs = []

    print("\n" + "=" * 70)
    print("STARTING IDW GENERATION FOR ALL VARIABLES AND ALL FOUR OUTPUTS")
    print("=" * 70)

    for var_name, csv_path in variable_csv_dict.items():
        try:
            print(f"\nProcessing variable: {var_name}")
            res_var = build_all_four_for_variable(
                var_name=var_name,
                df_stats_url=csv_path,
                loc_csv=loc_csv,
                out_gdb=out_gdb,
                map_name=map_name,
                clip_layer_name=clip_layer_name,
                xy_epsg=xy_epsg,
                cell_size=cell_size,
                power=power,
                clip_output=clip_output
            )
            all_results[var_name] = res_var
            print(f"✓ Finished {var_name}")

        except Exception as e:
            print(f"✗ FAILED variable {var_name}: {e}")
            failed_runs.append((var_name, str(e)))

            import traceback
            traceback.print_exc()

    print("\n" + "=" * 70)
    print("DONE")
    print("=" * 70)

    n_success_vars = len(all_results)
    n_total_vars = len(variable_csv_dict)

    print(f"\nVariables completed: {n_success_vars}/{n_total_vars}")

    print("\nSummary of successful outputs:")
    for var_name, var_dict in all_results.items():
        print(f"\n{var_name}:")
        for _, res in var_dict.items():
            print(f"  - {res['final_layer_name']}")

    if failed_runs:
        print("\nFailed variables:")
        for var_name, msg in failed_runs:
            print(f"  - {var_name}: {msg}")

    return all_results, failed_runs


# ============================================================
# CONFIG
# ============================================================
other_var_csvs = {
    "PRCP": r"./data_intermediate\prcp_station_stats.csv",
    "TMAX": r"./data_intermediate\tmax_station_stats.csv",
    "TMIN": r"./data_intermediate\tmin_station_stats.csv",
    "CSDI": r"./data_intermediate\csdi_station_stats.csv",
    "WSDI": r"./data_intermediate\wsdi_station_stats.csv",
    "SDII": r"./data_intermediate\sdii_station_stats.csv",
    "PCI":  r"./data_intermediate\pci_station_stats.csv",
    "AI":   r"./data_intermediate\ai_station_stats.csv"
}

other_var_csvs = {
    "PRCP": r"./data_intermediate\prcp_station_stats.csv",
    "TMAX": r"./data_intermediate\tmax_station_stats.csv",
   # "TMIN": r"./data_intermediate\tmin_station_stats.csv",
    "CSDI": r"./data_intermediate\csdi_station_stats.csv",
   # "WSDI": r"./data_intermediate\wsdi_station_stats.csv",
    "SDII": r"./data_intermediate\sdii_station_stats.csv",
   # "PCI":  r"./data_intermediate\pci_station_stats.csv",
    "AI":   r"./data_intermediate\ai_station_stats.csv"
}
# ============================================================
# RUN
# ============================================================
results_all, failed_all = build_all_variables_all_four(
    variable_csv_dict=other_var_csvs,
    loc_csv=loc_csv_updated,
    out_gdb=out_gdb_updated,
    map_name="Map2026_march",
    clip_layer_name="Eco region",
    xy_epsg=4326,
    cell_size=None,
    power=2,
    clip_output=True
)